# Loading and Running a Pre-trained LLM

## Changes from the tutorial

This notebook requires the modified `helper.py` and uses the `model_checkpoint_v2.orbax` produced by the modified L4 notebook. The new name keeps the masked-loss training run distinct from the tutorial checkpoint; the model architecture itself is unchanged.

Generation now performs top-k categorical sampling when temperature is greater than zero. In the tutorial, temperature was followed by `argmax`, so it could not affect the selected token. Use temperature `0` for deliberate greedy decoding, vary the seed for another reproducible sample, and increase top-k to admit more candidate tokens.

In [ ]:
from pathlib import Path

import jax
from flax import nnx
from helper import MiniGPT, generate_story
from IPython.display import display

In [ ]:
model = MiniGPT()

In [ ]:
display(model)

## Load the saved checkpoint

In [ ]:
import orbax
from jax.sharding import SingleDeviceSharding
from orbax import checkpoint

In [ ]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [ ]:
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

In [ ]:
nnx.state(model)

In [ ]:
# This must be the checkpoint produced by the updated L4 notebook.
checkpoint_path = Path.cwd() / "model_checkpoint_v2.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model,restored_state)

## Run inference

In [ ]:
def create_story(story_prompt, temperature, max_new_tokens, top_k, seed):
    return generate_story(
        model,
        story_prompt,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        top_k=top_k,
        seed=seed,
    )

In [ ]:
create_story("Once upon a time a big bear ", 0.8, 30, 50, 42)

## Interactive chat demo

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),
        gr.Slider(
            minimum=0, maximum=1.5, value=0.8, step=0.05, label="Temperature"
        ),
        gr.Slider(minimum=1, maximum=200, value=50, step=1, label="Max Tokens"),
        gr.Slider(minimum=1, maximum=200, value=50, step=1, label="Top-k"),
        gr.Number(value=42, precision=0, label="Random Seed"),
    ],
    outputs=["text"]
)

demo.launch(share=True)